# LLR Comparison: Base ESM2 vs LoRA-Finetuned

Loads the LoRA delta weights produced by `ESM_domainome_ft.ipynb`, recomputes per-domain
LLR matrices with both the un-finetuned base model and the finetuned model, and uploads a
single dictionary `{domain_id: {sequence, seq_offset, base_llr, ft_llr, delta_llr, ...}}`
to GCS as a gzipped pickle.


In [ ]:
import os
import io
import gzip
import json
import pickle
import warnings
import logging

import numpy as np
import pandas as pd
import torch
from tqdm import tqdm

from transformers import EsmForMaskedLM, EsmTokenizer
from peft import LoraConfig, get_peft_model
from google.cloud import storage

warnings.filterwarnings("ignore")
logging.getLogger("transformers").setLevel(logging.ERROR)

In [ ]:
# ── Constants (must match training run) ──────────────────────────────────────

AA_ORDER         = ['L', 'A', 'G', 'V', 'S', 'E', 'R', 'T', 'I', 'D',
                    'P', 'K', 'Q', 'N', 'F', 'Y', 'M', 'H', 'W', 'C']
FITNESS_AA_ORDER = list("ACDEFGHIKLMNPQRSTVWY")
MODEL_NAME       = "facebook/esm2_t33_650M_UR50D"
MAX_LEN          = 1022
LORA_DROPOUT     = 0.1

# ── Local + GCS layout (must match training run) ─────────────────────────────

OUTPUT_DIR = "finetuned_models_v2"
RUN_ID     = "global_finetune"
RUN_DIR    = os.path.join(OUTPUT_DIR, RUN_ID)
os.makedirs(RUN_DIR, exist_ok=True)

GCS_PROJECT           = ""   # e.g. "my-gcp-project"
GCS_BUCKET            = ""   # e.g. "my-bucket"
GCS_DOMAIN_DATA_BLOB  = ""   # e.g. "domainome/dict_fitness.pkl.gz"
GCS_FULL_PROTEIN_BLOB = ""   # e.g. "domainome/full_proteins.pkl.gz"
GCS_OUTPUT_PREFIX     = ""   # e.g. "domainome/finetuned_models_v2/" (must end with /)

# Where the LoRA weights and metadata produced by training live in GCS.
GCS_LORA_BLOB     = f"{GCS_OUTPUT_PREFIX}{RUN_ID}/best_model_lora.pth"
GCS_METADATA_BLOB = f"{GCS_OUTPUT_PREFIX}{RUN_ID}/metadata.json"

# Where the LLR comparison dict will be uploaded.
LLR_OUTPUT_NAME    = "llr_comparison.pkl.gz"
LLR_LOCAL_PATH     = os.path.join(RUN_DIR, LLR_OUTPUT_NAME)
GCS_LLR_OUTPUT_KEY = f"{GCS_OUTPUT_PREFIX}{RUN_ID}/{LLR_OUTPUT_NAME}"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
# ── GCS Helpers ──────────────────────────────────────────────────────────────

_gcs_client = storage.Client(project=GCS_PROJECT) if GCS_PROJECT else storage.Client()
_gcs_bucket = _gcs_client.bucket(GCS_BUCKET)


def gcs_download_pickle_gz(blob):
    data = blob.download_as_bytes()
    with gzip.GzipFile(fileobj=io.BytesIO(data), mode="rb") as gz:
        return pickle.load(gz)


def gcs_download_to_file(blob_path, local_path):
    blob = _gcs_bucket.blob(blob_path)
    os.makedirs(os.path.dirname(local_path), exist_ok=True)
    blob.download_to_filename(local_path)
    return local_path


def gcs_upload_file(blob_path, local_path, content_type="application/octet-stream"):
    blob = _gcs_bucket.blob(blob_path)
    blob.upload_from_filename(local_path, content_type=content_type)

In [ ]:
# ── Helpers (mirrored from training notebook) ────────────────────────────────

def truncate_to_domain(sequence, domain_start, domain_end, max_len=MAX_LEN):
    seq_len    = len(sequence)
    domain_mid = (domain_start + domain_end) // 2
    half       = max_len // 2
    start      = max(0, domain_mid - 1 - half)
    end        = start + max_len
    if end > seq_len:
        end   = seq_len
        start = max(0, end - max_len)
    return sequence[start:end], start


def get_LLR_scores(sequence, model, tokenizer, device):
    seq_list  = list(sequence)
    tokenized = tokenizer(sequence, padding=False, truncation=True,
                          max_length=1024, return_tensors="pt")
    input_ids = tokenized["input_ids"].to(device)

    logits    = model(input_ids).logits
    log_probs = torch.log_softmax(logits, dim=-1)
    wt_logits = log_probs[:, 1:-1, :].squeeze(0)

    wt_logits_df = pd.DataFrame(
        wt_logits[:, 4:24].cpu().detach().numpy(),
        columns=AA_ORDER,
        index=[f"{aa} {i+1}" for i, aa in enumerate(seq_list)]
    ).T

    wt_norm = np.diag(wt_logits_df.loc[[c.split(" ")[0] for c in wt_logits_df.columns]])
    LLR     = wt_logits_df - wt_norm
    return LLR, wt_logits, seq_list


def fitness_matrix_to_rows(fitness, dom_seq, position_offset=0):
    rows = []
    for i, wt_aa in enumerate(dom_seq):
        if i >= len(fitness):
            break
        row = fitness[i]
        for j, mut_aa in enumerate(FITNESS_AA_ORDER):
            if j >= len(row):
                continue
            val = row[j]
            if val is None:
                continue
            try:
                fval = float(val)
            except (TypeError, ValueError):
                continue
            if np.isnan(fval):
                continue
            if mut_aa == wt_aa:
                continue
            rows.append({
                "position":           position_offset + i + 1,
                "wt_aa":              wt_aa,
                "mut_aa":             mut_aa,
                "normalized_fitness": fval,
            })
    return rows


def build_lora_model(lora_r, lora_alpha):
    """Rebuild the exact ESM2 + LoRA architecture used during training."""
    model       = EsmForMaskedLM.from_pretrained(MODEL_NAME)
    lora_config = LoraConfig(
        task_type      = "FEATURE_EXTRACTION",
        r              = lora_r,
        lora_alpha     = lora_alpha,
        target_modules = ["query", "key", "value", "output.dense"],
        lora_dropout   = LORA_DROPOUT,
    )
    model = get_peft_model(model, lora_config)
    return model

In [ ]:
# ── Load metadata + LoRA weights from GCS (or local cache) ───────────────────

metadata_local = os.path.join(RUN_DIR, "metadata.json")
lora_local     = os.path.join(RUN_DIR, "best_model_lora.pth")

if not os.path.exists(metadata_local):
    print(f"Downloading metadata from gs://{GCS_BUCKET}/{GCS_METADATA_BLOB}")
    gcs_download_to_file(GCS_METADATA_BLOB, metadata_local)
else:
    print(f"Using local metadata: {metadata_local}")

if not os.path.exists(lora_local):
    print(f"Downloading LoRA weights from gs://{GCS_BUCKET}/{GCS_LORA_BLOB}")
    gcs_download_to_file(GCS_LORA_BLOB, lora_local)
else:
    print(f"Using local LoRA weights: {lora_local}")

with open(metadata_local) as f:
    metadata = json.load(f)

best_cfg = metadata["best_config"]
print(f"Best config -> lr={best_cfg['lr']}, lora_r={best_cfg['lora_r']}, "
      f"lora_alpha={best_cfg['lora_alpha']}")
print(f"Domains in run: {metadata.get('n_domains')}")

In [ ]:
# ── Recover per-domain sequences ─────────────────────────────────────────────
#
# Prefer the `domains` block embedded in metadata (already includes the truncated
# input sequence + seq_offset that training used). Fall back to rebuilding from
# the raw fitness/full-protein dicts in GCS if metadata doesn't carry them.

domains = {}

meta_domains = metadata.get("domains")
if isinstance(meta_domains, dict) and meta_domains:
    sample = next(iter(meta_domains.values()))
    if isinstance(sample, dict) and "sequence" in sample:
        for domain_id, dinfo in meta_domains.items():
            domains[domain_id] = {
                "uniprot_id": dinfo.get("uniprot_id"),
                "sequence":   dinfo["sequence"],
                "seq_offset": int(dinfo.get("seq_offset", 0)),
                "used_full_protein": bool(dinfo.get("used_full_protein", False)),
                "dom_start_in_protein": int(dinfo.get("dom_start_in_protein", 0)),
            }
        print(f"Recovered {len(domains)} domain sequences from metadata.json")

if not domains:
    print("metadata.json does not carry per-domain sequences — rebuilding from raw GCS dicts")
    full_protein_dict = {}
    if GCS_FULL_PROTEIN_BLOB:
        full_protein_dict = gcs_download_pickle_gz(
            _gcs_bucket.blob(GCS_FULL_PROTEIN_BLOB)
        )
        print(f"Loaded {len(full_protein_dict)} full proteins")
    fitness_dict = gcs_download_pickle_gz(_gcs_bucket.blob(GCS_DOMAIN_DATA_BLOB))
    print(f"Loaded {len(fitness_dict)} fitness entries")

    for domain_id, d in tqdm(fitness_dict.items(), desc="Rebuilding domains"):
        if not isinstance(d, dict):
            continue
        fitness    = d.get("fitness")
        uniprot_id = d.get("uniprot_id", domain_id.split("_")[0])
        dom_seq    = d.get("dom_seq")
        if fitness is None or dom_seq is None:
            continue

        full_seq             = None
        dom_start_in_protein = 0
        entry = full_protein_dict.get(uniprot_id)
        if entry:
            candidate = entry.get("sequence") if isinstance(entry, dict) else entry
            if isinstance(candidate, str):
                idx = candidate.find(dom_seq)
                if idx >= 0:
                    full_seq             = candidate
                    dom_start_in_protein = idx

        rows = fitness_matrix_to_rows(fitness, dom_seq,
                                      position_offset=dom_start_in_protein)
        if len(rows) < 10:
            continue

        seq_offset = 0
        if full_seq is not None:
            sequence = full_seq
            if len(sequence) > MAX_LEN:
                domain_start_1idx = dom_start_in_protein + 1
                domain_end_1idx   = dom_start_in_protein + len(dom_seq)
                sequence, seq_offset = truncate_to_domain(
                    sequence, domain_start_1idx, domain_end_1idx
                )
        else:
            sequence = dom_seq
            if len(sequence) > MAX_LEN:
                positions    = [r["position"] for r in rows]
                domain_start = min(positions)
                domain_end   = max(positions)
                sequence, seq_offset = truncate_to_domain(
                    sequence, domain_start, domain_end
                )

        domains[domain_id] = {
            "uniprot_id":           uniprot_id,
            "sequence":             sequence,
            "seq_offset":           seq_offset,
            "used_full_protein":    full_seq is not None,
            "dom_start_in_protein": dom_start_in_protein,
        }
    print(f"Rebuilt {len(domains)} domains from raw GCS dicts")

In [ ]:
# ── Load tokenizer + both models ─────────────────────────────────────────────

tokenizer = EsmTokenizer.from_pretrained(MODEL_NAME)

print("Loading base ESM2...")
base_model = EsmForMaskedLM.from_pretrained(MODEL_NAME).to(device)
base_model.eval()

print("Rebuilding LoRA architecture and loading finetuned weights...")
ft_model = build_lora_model(best_cfg["lora_r"], best_cfg["lora_alpha"])
lora_state = torch.load(lora_local, map_location="cpu")
missing, unexpected = ft_model.load_state_dict(lora_state, strict=False)
print(f"  loaded {len(lora_state)} LoRA tensors, {len(unexpected)} unexpected keys")
ft_model.to(device)
ft_model.eval()

In [ ]:
# ── Compute LLRs with both models for every domain ───────────────────────────

llr_dict = {}

with torch.no_grad():
    for domain_id, dinfo in tqdm(domains.items(), desc="Scoring domains"):
        sequence   = dinfo["sequence"]
        seq_offset = dinfo["seq_offset"]

        base_LLR, _, _ = get_LLR_scores(sequence, base_model, tokenizer, device)
        ft_LLR,   _, _ = get_LLR_scores(sequence, ft_model,   tokenizer, device)
        delta_LLR      = ft_LLR - base_LLR

        # Stored as plain numpy arrays + axis labels for portability; pickling
        # full DataFrames also works but balloons the file size unnecessarily.
        llr_dict[domain_id] = {
            "uniprot_id":           dinfo.get("uniprot_id"),
            "sequence":             sequence,
            "seq_offset":           seq_offset,
            "used_full_protein":    dinfo.get("used_full_protein", False),
            "dom_start_in_protein": dinfo.get("dom_start_in_protein", 0),
            "aa_order":             AA_ORDER,
            "positions":            list(base_LLR.columns),
            "base_llr":             base_LLR.to_numpy().astype(np.float32),
            "ft_llr":               ft_LLR.to_numpy().astype(np.float32),
            "delta_llr":            delta_LLR.to_numpy().astype(np.float32),
        }

print(f"Computed LLR comparisons for {len(llr_dict)} domains")

In [ ]:
# ── Quick sanity summary ─────────────────────────────────────────────────────

delta_means = []
delta_abs_means = []
for v in llr_dict.values():
    delta_means.append(float(np.nanmean(v["delta_llr"])))
    delta_abs_means.append(float(np.nanmean(np.abs(v["delta_llr"]))))

print(f"Mean ΔLLR across domains:     {np.mean(delta_means):+.4f}")
print(f"Mean |ΔLLR| across domains:   {np.mean(delta_abs_means):.4f}")
print(f"Max  |ΔLLR| in any domain:    {max(np.nanmax(np.abs(v['delta_llr'])) for v in llr_dict.values()):.4f}")

In [ ]:
# ── Save dictionary locally + upload to GCS ──────────────────────────────────

payload = {
    "run_id":      RUN_ID,
    "model_name":  MODEL_NAME,
    "best_config": best_cfg,
    "aa_order":    AA_ORDER,
    "n_domains":   len(llr_dict),
    "domains":     llr_dict,
}

with gzip.open(LLR_LOCAL_PATH, "wb", compresslevel=6) as gz:
    pickle.dump(payload, gz, protocol=pickle.HIGHEST_PROTOCOL)

size_mb = os.path.getsize(LLR_LOCAL_PATH) / 1e6
print(f"Wrote {LLR_LOCAL_PATH} ({size_mb:.1f} MB)")

print(f"Uploading to gs://{GCS_BUCKET}/{GCS_LLR_OUTPUT_KEY}")
gcs_upload_file(GCS_LLR_OUTPUT_KEY, LLR_LOCAL_PATH, content_type="application/gzip")
print("Done.")

## Reload the comparison dict

```python
import gzip, pickle, pandas as pd
from google.cloud import storage

blob = storage.Client().bucket("<bucket>").blob(
    "<prefix>/global_finetune/llr_comparison.pkl.gz"
)
with gzip.open(io.BytesIO(blob.download_as_bytes()), "rb") as gz:
    payload = pickle.load(gz)

entry = payload["domains"]["<domain_id>"]
base_df  = pd.DataFrame(entry["base_llr"],  index=entry["aa_order"], columns=entry["positions"])
ft_df    = pd.DataFrame(entry["ft_llr"],    index=entry["aa_order"], columns=entry["positions"])
delta_df = pd.DataFrame(entry["delta_llr"], index=entry["aa_order"], columns=entry["positions"])
```